# Test Backwards DeepFOC

**Table of contents**<a id='toc0_'></a>    
- 1. [Imports](#toc1_)    
- 2. [Setings](#toc2_)    
- 3. [Simultaneous Solve](#toc3_)    
- 4. [Backwards Solve](#toc4_)    
- 5. [Pure backward solve](#toc5_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=2
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## 1. <a id='toc1_'></a>[Imports](#toc0_)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import time
import numpy as np
import torch

In [3]:
import matplotlib.pyplot as plt
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.rcParams.update({'axes.grid':True,'grid.color':'black','grid.alpha':'0.25','grid.linestyle':'--'})
plt.rcParams.update({'font.size': 14})

In [4]:
from EconDLSolvers import choose_gpu, clean_solving_json
from DurablesModel import DurablesModelClass
clean_solving_json()

## 2. <a id='toc2_'></a>[Setings](#toc0_)

In [5]:
LOAD_SIMULT = False
K_time_simult = 1.0 
K_time = 1.0

par={'D':1,'KKT':True}

In [6]:
algoname = 'DeepFOC'
filename = '../output/DurablesModel_DeepFOC_1D.pt'

In [7]:
def print_R(model):
    model.simulate_R()
    R = model.sim.R.item()
    print(f'{R = :12.8f}')

## 3. <a id='toc3_'></a>[Simultaneous solve](#toc0_)

In [8]:
device = choose_gpu()

GPU 0: 44.40GB free [NVIDIA L40]
Best GPU: 0


In [9]:
if LOAD_SIMULT:
   
    model_simult = DurablesModelClass(load=filename,device=device)

else:

    model_simult = DurablesModelClass(
        algoname=algoname,device=device,
        par=par,
        train={'K_time':K_time_simult})
    
    model_simult.solve(do_print=True,do_print_all=False)

started solving: 2025-11-06 20:59:11


k =     0 of inf: sim.R = -504.05303955 [best: -504.05303955] [3.6 secs] [value_epochs =   0] [policy_epochs =  15] [  0.06 mins] [policy_lr = 1.0e-03]


k =    10 of inf: sim.R = -29.68570709 [best: -29.68570709] [3.3 secs] [value_epochs =   0] [policy_epochs =  15] [  0.14 mins] [policy_lr = 1.0e-03]


k =    20 of inf: sim.R = -29.24090767 [best: -29.24090767] [3.3 secs] [value_epochs =   0] [policy_epochs =  15] [  0.24 mins] [policy_lr = 1.0e-03]


k =    30 of inf: sim.R = -29.06369019 [best: -29.06369019] [3.3 secs] [value_epochs =   0] [policy_epochs =  15] [  0.33 mins] [policy_lr = 1.0e-03]


...

k =    80 of inf: sim.R = -28.97818947 [best: -28.97818947] [3.4 secs] [value_epochs =   0] [policy_epochs =  15] [  0.82 mins] [policy_lr = 9.9e-04]


k =    90 of inf: sim.R = -28.97661018 [best: -28.97661018] [3.4 secs] [value_epochs =   0] [policy_epochs =  15] [  0.91 mins] [policy_lr = 9.9e-04]


k =   100 of inf: sim.R = -28.97509193 [best: -28.97509193] [3.4 secs] [value_epochs =   0] [policy_epochs =  15] [  1.00 mins] [policy_lr = 9.9e-04]


Terminating after 101 episodes, max time 1.0 mins reached


R = -28.9751, time = 1.0 mins, iter = 101, policy epochs = 14.03, value epochs = 0.00


In [10]:
print_R(model_simult)

R = -28.97509193


## 4. <a id='toc4_'></a>[Backwards refinement](#toc0_)

In [11]:
train = {}
train['Nneurons_policy_t'] = np.array([50,50])
train['N'] = 500_000
train['batch_size'] = 10_000
train['K_time'] = K_time

In [12]:
model_refine = DurablesModelClass(
    algoname=f'{algoname}Backward',device=device,
    par=par,train=train
)

In [13]:
model_refine.solve(do_print=True,model_simult=model_simult)
print(f'{model_refine.info["time"]:.1f} mins')

started solving: 2025-11-06 21:00:12


t =  29


 epoch =     0: 1.9e-04

 epoch =    10: 8.8e-05 time limit reached [  2.0 secs]
t =  28


 epoch =     0: 1.4e-01

...

 epoch =     0: 1.1e-04

 epoch =     1: 8.5e-05 time limit reached [  3.1 secs]
t =   0


 epoch =     0: 2.1e-04

 epoch =     1: 9.5e-05 time limit reached [  3.1 secs]


193.1 mins


R:

In [14]:
print_R(model_simult)
print_R(model_refine)

R = -28.97509193


R = -28.97341919


## 5. <a id='toc5_'></a>[Pure backwards](#toc0_)

In [15]:
train = {}
train['use_simult_in_backward'] = False
train['Nneurons_policy_t'] = np.array([50,50])
train['N'] = 500_000
train['batch_size'] = 10_000
train['K_time'] = K_time

In [16]:
model_pure = DurablesModelClass(
    algoname=f'{algoname}Backward',device=device,
    par=par,train=train
)

In [17]:
model_pure.solve(do_print=True)
print(f'{model_pure.info["time"]:.1f} mins')

started solving: 2025-11-06 21:01:51


t =  29
 epoch =     0: 3.4e+00

 epoch =    10: 2.9e-02

 epoch =    13: 1.6e-02 time limit reached [  2.0 secs]
t =  28


 epoch =     0: 3.7e-01

...

 epoch =     7: 9.1e-04 time limit reached [  2.0 secs]
t =   1


 epoch =     0: 2.9e-03

 epoch =     7: 8.1e-04 time limit reached [  2.0 secs]
t =   0


 epoch =     0: 2.7e-03

 epoch =     7: 7.1e-04 time limit reached [  2.0 secs]
131.6 mins


In [18]:
print_R(model_simult)
print_R(model_pure)

R = -28.97509193
R = -29.14447975
